# CircuitMind — LangChain Requirement Analysis Agent

A small LangChain + Groq research prototype for turning a hardware idea into validated requirements through a short QA conversation.

## 0. Environment Setup

In [1]:
%pip install -U "langchain>=0.3" "langchain-groq>=0.2" "pydantic>=2.0" "gradio>=4.0" matplotlib


Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
import ast
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from dotenv import load_dotenv
_cwd = Path.cwd()
_repo_root = _cwd if (_cwd / "ai_engine").is_dir() else _cwd.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


## 2. Configuration

In [3]:
# Load the AI engine's local environment regardless of whether Jupyter starts in the repo root or ai_engine/.
for env_path in (Path.cwd() / ".env", Path.cwd() / "ai_engine" / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path, override=False)
        break

MODEL_NAME = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TEMPERATURE = 0.2
MAX_FOLLOW_UP_QUESTIONS = 7

if not GROQ_API_KEY:
    raise EnvironmentError("Set GROQ_API_KEY before running the LangChain cells.")

llm = ChatGroq(
    model=MODEL_NAME,
    groq_api_key=GROQ_API_KEY,
    temperature=TEMPERATURE,
    max_retries=2,
)

print(f"Model: {MODEL_NAME}")
print("Provider: Groq")
print(f"Temperature: {TEMPERATURE}")


Model: llama-3.3-70b-versatile
Provider: Groq
Temperature: 0.2


## 3. Pydantic Schema

In [4]:
class HardwareRequirements(BaseModel):
    """Architecture-oriented requirements for the user's hardware project."""

    model_config = ConfigDict(extra="forbid")

    project_name: str | None = Field(default=None, description="Name of the user's hardware project")
    category: str | None = None
    objective: str | None = None
    # The unions keep Groq's tool schema tolerant of scalar/dict variants.
    # Validators below normalize them into the clean list/string output.
    target_users: list[str] | str | dict[str, Any] | None = None
    functional_requirements: list[str] | str | dict[str, Any] | None = None
    hardware_inputs: list[str] | str | dict[str, Any] | None = None
    hardware_outputs: list[str] | str | dict[str, Any] | None = None
    connectivity: list[str] | str | dict[str, Any] | None = None
    supported_platforms: list[str] | str | dict[str, Any] | None = None
    power_requirements: str | list[str] | dict[str, Any] | None = None
    physical_constraints: list[str] | str | dict[str, Any] | None = None
    performance_requirements: list[str] | str | dict[str, Any] | None = None
    safety_compliance: list[str] | str | dict[str, Any] | None = None
    budget: str | int | float | None = None

    @field_validator("project_name", "category", "objective", "power_requirements", "budget", mode="before")
    @classmethod
    def normalize_scalar(cls, value):
        if value is None:
            return None
        if isinstance(value, str):
            text = value.strip()
            if text.lower() in {"", "null", "none", "unknown"}:
                return None
            if text.startswith("[") and text.endswith("]"):
                try:
                    parsed = json.loads(text)
                except json.JSONDecodeError:
                    try:
                        parsed = ast.literal_eval(text)
                    except (ValueError, SyntaxError):
                        parsed = text
                if isinstance(parsed, list):
                    return str(parsed[0]) if parsed and parsed[0] is not None else None
        return str(value)

    @field_validator(
        "target_users", "functional_requirements", "hardware_inputs",
        "hardware_outputs", "connectivity", "supported_platforms",
        "physical_constraints",
        "performance_requirements", "safety_compliance", mode="before"
    )
    @classmethod
    def normalize_list(cls, value):
        if value is None:
            return None
        if isinstance(value, dict):
            value = [f"{key}: {item}" for key, item in value.items() if item is not None]
        elif isinstance(value, str):
            text = value.strip()
            if text.lower() in {"", "null", "none", "unknown", "[]"}:
                return None
            if text.startswith("[") and text.endswith("]"):
                try:
                    value = json.loads(text)
                except json.JSONDecodeError:
                    try:
                        value = ast.literal_eval(text)
                    except (ValueError, SyntaxError):
                        value = [text]
            else:
                value = [text]
        values = [str(item).strip() for item in value if item is not None and str(item).strip()]
        return values or None


class InterviewResponse(BaseModel):
    """One question or the final validated requirements object."""

    model_config = ConfigDict(extra="forbid")

    status: Literal["question", "complete"]
    question: str | None = None
    options: list[str] | None = None
    requirements: HardwareRequirements | None = None


class QuestionOptions(BaseModel):
    options: list[str] = Field(min_length=2, max_length=4)

HardwareRequirements.model_json_schema()


{'additionalProperties': False,
 'description': "Architecture-oriented requirements for the user's hardware project.",
 'properties': {'project_name': {'anyOf': [{'type': 'string'},
    {'type': 'null'}],
   'default': None,
   'description': "Name of the user's hardware project",
   'title': 'Project Name'},
  'category': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Category'},
  'objective': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Objective'},
  'target_users': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'},
    {'type': 'string'},
    {'additionalProperties': True, 'type': 'object'},
    {'type': 'null'}],
   'default': None,
   'title': 'Target Users'},
  'functional_requirements': {'anyOf': [{'items': {'type': 'string'},
     'type': 'array'},
    {'type': 'string'},
    {'additionalProperties': True, 'type': 'object'},
    {'type': 'null'}],
   'default': None,
   'title': 'Functional Requ

## 4. LangChain Prompt

In [5]:
SYSTEM_PROMPT = """
For every question, provide 2-4 concise options whenever reasonable, especially for power, budget, connectivity, platform, and performance. Use null options only for genuinely open-ended questions.\n
You are dunkai's Requirement Analysis Agent. dunkai is the software product, not the user's hardware project.

Architecture-first completion rule: conduct at least five and at most seven useful interview turns before status complete. Use high-yield grouped questions instead of one question per schema field. Cover: (1) user workflow and main functions, (2) physical inputs and sensing, (3) physical outputs and interaction, (4) connectivity, processing location, and host platforms, and (5) power, battery life, physical constraints, performance, and safety. Combine related topics into one concise project-specific question. Do not invent exact components or specifications.

Conduct a 5–7 turn, project-specific requirements interview. Ask exactly one concise grouped follow-up question per turn. A grouped question may ask several closely related details that together affect architecture. Do not repeat questions or ask narrow low-value questions. Treat the entire conversation as cumulative state: preserve every fact from earlier user answers, merge the latest answer into the existing requirements, and never replace known values with null. Map answers explicitly into the appropriate fields, especially hardware_inputs, hardware_outputs, functional_requirements, connectivity, and power_requirements. Only return complete after the minimum five interview turns, unless the conversation already contains five clear user answers. When a question has common discrete answers, provide 2–4 concise options; otherwise set options to null. The user may always provide a custom answer.

Ask only about information that can affect architecture, hardware inputs/outputs, connectivity, supported platforms, power, physical constraints, performance, safety, or budget. Do not ask generic questions when a project-specific question is possible. Do not repeat answered questions. If the latest answer is vague or does not answer the previous question, clarify it instead of changing the project.

Never hallucinate. Do not change the project domain. Unknown values must be null. Do not recommend components, design circuits, or generate firmware. Ask at least five and no more than seven questions total.

Return only the structured response represented by the Pydantic schema. For complete responses, set question and options to null.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
structured_llm = llm.with_structured_output(InterviewResponse)
interview_chain = prompt | structured_llm

option_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 2 to 4 useful answer choices for the question. Choices must be specific to the hardware project and question. Do not answer the question. Return only the options field."),
    ("human", "Question: {question}"),
])
option_chain = option_prompt | llm.with_structured_output(QuestionOptions)


## 5. LangChain QA Chain

In [6]:
def to_langchain_history(history: list[Any] | None) -> list[Any]:
    """Support both current and older Gradio history formats."""
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            role, content = item.get("role"), item.get("content")
            if role == "user" and content:
                messages.append(HumanMessage(content=str(content)))
            elif role == "assistant" and content:
                messages.append(AIMessage(content=str(content)))
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0]: messages.append(HumanMessage(content=str(item[0])))
            if item[1]: messages.append(AIMessage(content=str(item[1])))
    return messages[-14:]


def _asked_question_count(history: list[Any] | None) -> int:
    count = 0
    for item in history or []:
        content = item.get("content") if isinstance(item, dict) and item.get("role") == "assistant" else (item[1] if isinstance(item, (list, tuple)) and len(item) == 2 else None)
        if isinstance(content, str) and content.strip() and not content.lstrip().startswith("{"):
            count += 1
    return count


def run_interview(user_input: str, history: list[Any] | None = None) -> InterviewResponse:
    if not user_input or not user_input.strip():
        raise ValueError("Please enter a hardware project idea.")
    try:
        asked = _asked_question_count(history)
        turn_instruction = (
            f"This is follow-up question {asked + 1} of 5 minimum. Ask a grouped architecture question; do not complete yet.\n"
            if asked < 5 else "The minimum five questions have been asked; complete only if architecture-critical details are sufficient.\n"
        )
        current_input = turn_instruction + "\nCURRENT USER ANSWER:\n" + user_input.strip()
        result = interview_chain.invoke({"history": to_langchain_history(history), "input": current_input})
        response = InterviewResponse.model_validate(result)
        if response.status == "question" and response.question and not response.options:
            try:
                generated = option_chain.invoke({"question": response.question})
                response = response.model_copy(update={"options": QuestionOptions.model_validate(generated).options})
            except Exception:
                pass
        return response
    except ValidationError:
        raise
    except Exception as exc:
        raise RuntimeError(f"LangChain/Groq interview failed: {exc}") from exc


def respond(user_input: str, history: list[Any] | None = None) -> str:
    result = run_interview(user_input, history)
    if result.status == "question":
        return result.question or "Please provide one more project detail."
    return result.requirements.model_dump_json(indent=2, exclude_none=True)


## 6. Gradio Chat UI

In [7]:
import gradio as gr


def _content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                text = block.get("text") or block.get("content")
                if text is not None:
                    parts.append(str(text))
            elif content is not None:
                parts.append(str(block))
        return "".join(parts)
    return "" if content is None else str(content)


def as_chat_messages(history):
    # Normalize Gradio 6 history to strict message dictionaries.\n
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            content = _content_to_text(content)
            if role in {"user", "assistant", "system"} and content:
                messages.append({"role": role, "content": content})
        elif hasattr(item, "role") and hasattr(item, "content"):
            role = getattr(item, "role", None)
            content = getattr(item, "content", None)
            content = _content_to_text(content)
            if role in {"user", "assistant", "system"} and content:
                messages.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0] is not None:
                messages.append({"role": "user", "content": str(item[0])})
            if item[1] is not None:
                messages.append({"role": "assistant", "content": str(item[1])})
    return messages


def submit_answer(message, selected_option, custom_answer, history):
    answer = (custom_answer or selected_option or message or "").strip()
    chat_history = as_chat_messages(history)
    if not answer:
        return chat_history, gr.update(choices=[], value=None, visible=False), gr.update(value="", visible=False), ""

    result = run_interview(answer, chat_history)
    if result.status == "question":
        bot_message = result.question or "Please provide one more project detail."
        choices = result.options or []
        return (
            chat_history + [{"role": "user", "content": answer}, {"role": "assistant", "content": bot_message}],
            gr.update(choices=choices, value=None, visible=bool(choices)),
            gr.update(value="", visible=True),
            "",
        )

    return (
        chat_history + [{"role": "user", "content": answer}, {"role": "assistant", "content": result.requirements.model_dump_json(indent=2, exclude_none=True)}],
        gr.update(choices=[], value=None, visible=False),
        gr.update(value="", visible=False),
        "",
    )


with gr.Blocks(title="dunkai Requirement Agent") as requirement_app:
    gr.Markdown("## dunkai Requirement Agent")
    chatbot = gr.Chatbot( label="Conversation")
    message = gr.Textbox(label="Project idea or answer", placeholder="Example: I want to build a wireless gamepad.", lines=2)
    options = gr.Radio(label="Choose an option", choices=[], visible=False)
    custom_answer = gr.Textbox(label="Custom answer", placeholder="Or type your own answer here...", visible=False)
    submit = gr.Button("Submit", variant="primary")
    submit.click(
        submit_answer,
        inputs=[message, options, custom_answer, chatbot],
        outputs=[chatbot, options, custom_answer, message],
    )
    message.submit(
        submit_answer,
        inputs=[message, options, custom_answer, chatbot],
        outputs=[chatbot, options, custom_answer, message],
    )

# Legacy requirement-only app retained for reference; integrated app is launched below.


c:\Rayyan\Rayyan Development\circuit_mind_dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import matplotlib.pyplot as plt
import gradio as gr


class EvaluationScore(BaseModel):
    """Structured quality assessment for one agent response."""

    model_config = ConfigDict(extra="forbid")

    overall_score: int = Field(ge=1, le=5)
    relevance_score: int = Field(ge=1, le=5)
    groundedness_score: int = Field(ge=1, le=5)
    schema_score: int = Field(ge=1, le=5)
    qa_quality_score: int = Field(ge=1, le=5)
    strengths: list[str] = []
    weaknesses: list[str] = []
    hallucinations: list[str] = []


judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an impartial evaluator for dunkai's hardware requirement-analysis agent.
Score the response against the user's original idea. Reward relevant, grounded extraction and useful, non-repetitive QA. Penalize invented values, domain changes, generic or repeated questions, and schema violations.
Use 1 for poor and 5 for excellent. Return only the structured evaluation.
"""),
    ("human", "Original idea:\n{idea}\n\nAgent response:\n{agent_response}"),
])
judge_chain = judge_prompt | llm.with_structured_output(EvaluationScore)

evaluation_examples = [
    "I want to build an air purifier.",
    "I want to build a wireless gaming controller for PC and mobile.",
    "Build a solar-powered smart irrigation system for a small farm.",
    "Create a portable wheat disease detector for farmers.",
]


def evaluate_examples():
    rows = []
    for idea in evaluation_examples:
        response = run_interview(idea, [])
        if response.status == "question":
            agent_output = response.question or ""
            response_type = "question"
        else:
            agent_output = response.requirements.model_dump_json(indent=2, exclude_none=True)
            response_type = "complete"
        score = judge_chain.invoke({"idea": idea, "agent_response": agent_output})
        rows.append({
            "example": idea,
            "response_type": response_type,
            "overall": score.overall_score,
            "relevance": score.relevance_score,
            "groundedness": score.groundedness_score,
            "schema": score.schema_score,
            "qa_quality": score.qa_quality_score,
            "feedback": "; ".join(score.weaknesses or score.strengths),
            "hallucinations": "; ".join(score.hallucinations),
        })
    return rows


def evaluation_chart(rows):
    labels = [f"Example {index + 1}" for index in range(len(rows))]
    scores = [row["overall"] for row in rows]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, scores, color="#4f46e5")
    ax.set_ylim(0, 5)
    ax.set_ylabel("Overall score (1–5)")
    ax.set_title("dunkai Requirement Agent Evaluation")
    ax.grid(axis="y", alpha=0.25)
    for index, score in enumerate(scores):
        ax.text(index, score + 0.08, str(score), ha="center")
    fig.tight_layout()
    return fig


def run_evaluation_dashboard():
    rows = evaluate_examples()
    average = sum(row["overall"] for row in rows) / len(rows)
    summary = f"Average overall score: {average:.2f}/5 across {len(rows)} examples."
    table = [[row[key] for key in ["example", "response_type", "overall", "relevance", "groundedness", "schema", "qa_quality", "feedback", "hallucinations"]] for row in rows]
    return summary, table, evaluation_chart(rows)


with gr.Blocks(title="dunkai Evaluation Dashboard") as evaluation_app:
    gr.Markdown("## dunkai Agent Evaluation")
    run_button = gr.Button("Run evaluation", variant="primary")
    summary = gr.Markdown()
    results_table = gr.Dataframe(
        headers=["Example", "Type", "Overall", "Relevance", "Groundedness", "Schema", "QA quality", "Feedback", "Hallucinations"],
        interactive=False,
        wrap=True,
    )
    chart = gr.Plot(label="Overall scores")
    run_button.click(run_evaluation_dashboard, outputs=[summary, results_table, chart])

# Legacy evaluator app retained for reference; integrated app is launched below.


## Connected Requirement → Architecture → Evaluation UI

Run the next cell to use the Requirement Agent interview. When the interview reaches `complete`, the JSON is saved and passed automatically to the Architecture Agent and LLM evaluator.

In [9]:
from pathlib import Path
import importlib
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import ai_engine.architecture_agent as _architecture_agent
_architecture_agent = importlib.reload(_architecture_agent)

CATEGORY_COLUMNS = {'Power':0,'Input':1,'Sensor':1,'Processing':2,'Communication':3,'Network':3,'Storage':3,'Output':4}

def layout_architecture(graph, horizontal_spacing=320, vertical_spacing=160):
    flow = json.loads(json.dumps(graph))
    rows = {}
    for node in flow['nodes']:
        column = CATEGORY_COLUMNS.get(node['data']['category'], 2)
        row = rows.get(column, 0)
        rows[column] = row + 1
        node['position'] = {'x':column * horizontal_spacing, 'y':row * vertical_spacing}
    return flow

class ArchitectureJudge(BaseModel):
    overall_score: int = Field(ge=1, le=5)
    relevance_score: int = Field(ge=1, le=5)
    groundedness_score: int = Field(ge=1, le=5)
    graph_quality_score: int = Field(ge=1, le=5)
    verdict: Literal['PASS','REVIEW']
    strengths: list[str] = Field(default_factory=list)
    weaknesses: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)

architecture_judge_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a senior embedded-systems architecture reviewer. Judge the Architecture Agent output against the Requirement Agent JSON. Reward complete, relevant, vendor-neutral technology-level architecture. Penalize missing subsystems, unsupported interfaces, exact vendor/IC selections, weak assumptions, and missing warnings. Return only the structured evaluation.'),
    ('human', 'Requirement Agent JSON:\n{requirements}\n\nArchitecture Agent JSON:\n{architecture}')
])

def build_and_evaluate(requirements):
    architecture_result = _architecture_agent.build_architecture(requirements)
    graph = architecture_result.get('architecture_graph', architecture_result.get('react_flow'))
    react_flow = layout_architecture(graph)
    architecture_result = dict(architecture_result, architecture_graph=graph, react_flow=react_flow)
    api_key = os.getenv('GROQ_API_KEY')
    if not api_key:
        raise RuntimeError('GROQ_API_KEY is missing. Set it before completing the Requirement Agent interview.')
    model = ChatGroq(model=os.getenv('GROQ_MODEL','llama-3.3-70b-versatile'), groq_api_key=api_key, temperature=0, max_retries=2)
    judge = architecture_judge_prompt | model.with_structured_output(ArchitectureJudge)
    evaluation = judge.invoke({'requirements':json.dumps(requirements, indent=2), 'architecture':json.dumps(architecture_result, indent=2)}).model_dump()
    Path('requirement_output.json').write_text(json.dumps(requirements, indent=2), encoding='utf-8')
    Path('architecture_output.json').write_text(json.dumps(architecture_result, indent=2), encoding='utf-8')
    Path('architecture_evaluation.json').write_text(json.dumps(evaluation, indent=2), encoding='utf-8')
    return architecture_result, evaluation

def format_pipeline(requirements, architecture, evaluation):
    requirement_rows = [[key, ', '.join(value) if isinstance(value, list) else str(value)] for key, value in requirements.items()]
    model = architecture.get('architecture_model', {})
    nodes = architecture['react_flow']['nodes']
    edges = architecture['react_flow']['edges']
    architecture_summary = f"### Architecture Overview\n\n**Processing unit:** {model.get('processing_unit', 'Not specified')}  |  **Nodes:** {len(nodes)}  |  **Connections:** {len(edges)}"
    node_rows = [[n['data']['label'], n['data']['category'], 'Yes' if n['data'].get('inferred') else 'No', ', '.join(n['data'].get('sourceRequirements', []))] for n in nodes]
    edge_rows = [[e['source'], e.get('data', {}).get('interface', e.get('label', '')), e['target']] for e in edges]
    evaluation_summary = f"### LLM Evaluation: {evaluation.get('verdict', 'REVIEW')}\n\n**Overall:** {evaluation.get('overall_score', '-')}/5  |  **Relevance:** {evaluation.get('relevance_score', '-')}/5  |  **Groundedness:** {evaluation.get('groundedness_score', '-')}/5  |  **Graph quality:** {evaluation.get('graph_quality_score', '-')}/5"
    feedback = ([["Strength", item] for item in evaluation.get('strengths', [])] + [["Weakness", item] for item in evaluation.get('weaknesses', [])] + [["Warning", item] for item in evaluation.get('warnings', [])]) or [["Feedback", "No additional feedback"]]
    return requirement_rows, architecture_summary, node_rows, edge_rows, evaluation_summary, feedback

def submit_connected(message, selected_option, custom_answer, history):
    answer = (custom_answer or selected_option or message or '').strip()
    chat_history = as_chat_messages(history)
    empty = (chat_history, gr.update(choices=[], value=None, visible=False), gr.update(value='', visible=False), '', None, '', [], [], '', [], None, None, None)
    if not answer:
        return empty
    response = run_interview(answer, chat_history)
    if response.status == 'question':
        question = response.question or 'Please provide one more project detail.'
        choices = response.options or []
        return (chat_history + [{'role':'user','content':answer},{'role':'assistant','content':question}], gr.update(choices=choices, value=None, visible=bool(choices)), gr.update(value='', visible=True), '', None, '', [], [], '', [], None, None, None)
    requirements = response.requirements.model_dump(exclude_none=True)
    architecture, evaluation = build_and_evaluate(requirements)
    requirement_rows, architecture_summary, node_rows, edge_rows, evaluation_summary, feedback = format_pipeline(requirements, architecture, evaluation)
    final_message = 'Requirements complete. Architecture and evaluation are shown below.'
    return (chat_history + [{'role':'user','content':answer},{'role':'assistant','content':final_message}], gr.update(choices=[], value=None, visible=False), gr.update(value='', visible=False), '', requirement_rows, architecture_summary, node_rows, edge_rows, evaluation_summary, feedback, requirements, architecture, evaluation)

with gr.Blocks(title='dunkai Connected Agents') as connected_app:
    gr.Markdown('# Requirement Agent → Architecture Agent → Evaluation')
    gr.Markdown('Complete the interview. Results are shown as readable summaries and tables; raw JSON remains available below for downstream use.')
    chatbot = gr.Chatbot(label='Requirement Agent interview')
    message = gr.Textbox(label='Project idea or answer', lines=2)
    options = gr.Radio(label='Choose an option', choices=[], visible=False)
    custom_answer = gr.Textbox(label='Custom answer', visible=False)
    submit = gr.Button('Submit', variant='primary')
    gr.Markdown('## Stored Requirements')
    requirement_table = gr.Dataframe(headers=['Requirement field','Value'], interactive=False)
    gr.Markdown('## Architecture')
    architecture_summary = gr.Markdown()
    with gr.Row():
        nodes_table = gr.Dataframe(headers=['Node','Category','Inferred','Source requirements'], interactive=False)
        edges_table = gr.Dataframe(headers=['Source','Interface','Target'], interactive=False)
    evaluation_summary = gr.Markdown()
    feedback_table = gr.Dataframe(headers=['Type','Feedback'], interactive=False)
    with gr.Accordion('Raw JSON outputs', open=False):
        raw_requirements = gr.JSON(label='Requirement Agent JSON')
        raw_architecture = gr.JSON(label='Architecture Agent / React Flow JSON')
        raw_evaluation = gr.JSON(label='LLM evaluation JSON')
    outputs = [chatbot, options, custom_answer, message, requirement_table, architecture_summary, nodes_table, edges_table, evaluation_summary, feedback_table, raw_requirements, raw_architecture, raw_evaluation]
    submit.click(submit_connected, inputs=[message, options, custom_answer, chatbot], outputs=outputs)
    message.submit(submit_connected, inputs=[message, options, custom_answer, chatbot], outputs=outputs)
connected_app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
